<a href="https://colab.research.google.com/github/smrkrs/ArabicPython/blob/main/HW4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Задача 25 (паттерн Factory). Аэропорт регулярно пополняется разными типами самолётов — пассажирскими, грузовыми, чартерными. Создавать их напрямую через конструкторы неудобно: код разрастается, легко ошибиться. Создайте класс AircraftFactory с методом create(kind, **kwargs), который по строке "passenger", "cargo" или "charter" создаёт соответствующий объект и возвращает его. Неизвестный тип должен вызывать ValueError. Продемонстрируйте работу фабрики: создайте по одному самолёту каждого типа через фабрику. Объясните, почему фабрика удобнее прямого вызова конструкторов и как этот паттерн связан с принципом Open/Closed (см. задачу 35).

In [ ]:
class PassengerAircraft:
    def __init__(self, name, seats):
        self.name = name
        self.seats = seats


class CargoAircraft:
    def __init__(self, name, capacity):
        self.name = name
        self.capacity = capacity


class CharterAircraft:
    def __init__(self, name, customer):
        self.name = name
        self.customer = customer


class AircraftFactory:
    @staticmethod
    def create(kind, **kwargs):
        if kind == "passenger":
            return PassengerAircraft(**kwargs)
        elif kind == "cargo":
            return CargoAircraft(**kwargs)
        elif kind == "charter":
            return CharterAircraft(**kwargs)
        else:
            raise ValueError("Неизвестный тип самолёта")


passenger = AircraftFactory.create("passenger", name="П-1", seats=180)
cargo = AircraftFactory.create("cargo", name="Г-1", capacity=20)
charter = AircraftFactory.create("charter", name="Ч-1", customer="Турфирма")

print("Пассажирский:", passenger.name, "мест:", passenger.seats)
print("Грузовой:", cargo.name, "тонн:", cargo.capacity)
print("Чартерный:", charter.name, "заказчик:", charter.customer)

try:
    AircraftFactory.create("unknown")
except ValueError as error:
    print(error)

Пассажирский: П-1 мест: 180
Грузовой: Г-1 тонн: 20
Чартерный: Ч-1 заказчик: Турфирма
Неизвестный тип самолёта


Задача 26 (паттерн Observer). Пассажиры хотят получать уведомления об изменениях времени вылета или открытии онлайн-регистрации на их рейс. Реализуйте паттерн «Наблюдатель»: класс Flight хранит список подписчиков (Passenger) и при изменении времени вылета уведомляет всех через метод notify(). Продемонстрируйте, что при изменении времени все пассажиры получают уведомление. Дополнительно реализуйте unsubscribe() и покажите, что после отписки пассажир больше не получает уведомлений. Объясните, где в реальных проектах используется этот паттерн (GUI, событийные системы, мессенджеры, pub/sub) и как он помогает избежать жёсткой связи между объектами.

In [ ]:
class Passenger:
    def __init__(self, name):
        self.name = name

    def update(self, message):
        print(f"{self.name}: {message}")


class Flight:
    def __init__(self, number, departure_time):
        self.number = number
        self.departure_time = departure_time
        self.subscribers = []

    def subscribe(self, passenger):
        if passenger not in self.subscribers:
            self.subscribers.append(passenger)

    def unsubscribe(self, passenger):
        if passenger in self.subscribers:
            self.subscribers.remove(passenger)

    def notify(self, message):
        for passenger in self.subscribers:
            passenger.update(message)

    def change_departure_time(self, new_time):
        self.departure_time = new_time
        self.notify(f"Время вылета рейса {self.number} изменено на {new_time}.")

    def open_check_in(self):
        self.notify(f"Открыта онлайн-регистрация на рейс {self.number}.")


flight = Flight("SU123", "12:00")
passenger1 = Passenger("Анна")
passenger2 = Passenger("Иван")

flight.subscribe(passenger1)
flight.subscribe(passenger2)

print("Первое изменение времени:")
flight.change_departure_time("13:00")

flight.unsubscribe(passenger2)

print("\nИзменение времени после отписки Ивана:")
flight.change_departure_time("14:00")

print("\nОткрытие онлайн-регистрации:")
flight.open_check_in()

Первое изменение времени:
Анна: Время вылета рейса SU123 изменено на 13:00.
Иван: Время вылета рейса SU123 изменено на 13:00.

Изменение времени после отписки Ивана:
Анна: Время вылета рейса SU123 изменено на 14:00.

Открытие онлайн-регистрации:
Анна: Открыта онлайн-регистрация на рейс SU123.


Задача 27 (принцип SOLID — SRP). Класс Airport со временем разросся: он и хранит рейсы, и регистрирует пассажиров, и отправляет уведомления, и формирует отчёты. Это нарушает принцип единственной ответственности (SRP) — у класса должна быть одна причина для изменения. Разбейте класс Airport на несколько классов: FlightStorage (хранение рейсов и поиск по номеру), CheckInService (регистрация пассажиров, ведение учёта), NotificationService (уведомления). Продемонстрируйте работу собранных сервисов на сценарии «пассажир зарегистрировался на рейс». Объясните, почему это лучше одного большого класса, как понять, что у класса «одна причина для изменения», и приведите примеры «классов-богов» из реальной жизни.

In [ ]:
class FlightStorage:
    def __init__(self):
        self.flights = {}

    def add_flight(self, flight_number, destination):
        self.flights[flight_number] = destination

    def find_flight(self, flight_number):
        return self.flights.get(flight_number)


class CheckInService:
    def __init__(self, flight_storage):
        self.flight_storage = flight_storage
        self.registrations = {}

    def register(self, flight_number, passenger_name):
        if self.flight_storage.find_flight(flight_number) is None:
            raise ValueError("Рейс не найден")

        if flight_number not in self.registrations:
            self.registrations[flight_number] = []

        if passenger_name in self.registrations[flight_number]:
            raise ValueError("Пассажир уже зарегистрирован")

        self.registrations[flight_number].append(passenger_name)

    def get_passengers(self, flight_number):
        return self.registrations.get(flight_number, [])


class NotificationService:
    def send(self, passenger_name, message):
        print(f"{passenger_name}: {message}")


storage = FlightStorage()
storage.add_flight("SU123", "Сочи")

check_in = CheckInService(storage)
notifications = NotificationService()

flight_number = "SU123"
passenger_name = "Анна"

check_in.register(flight_number, passenger_name)
notifications.send(passenger_name, f"Вы зарегистрированы на рейс {flight_number}.")

print("Зарегистрированные пассажиры:", check_in.get_passengers(flight_number))

Анна: Вы зарегистрированы на рейс SU123.
Зарегистрированные пассажиры: ['Анна']


Задача 28 (композиция vs наследование). Бизнес-класс (BusinessPassenger) похож на обычного пассажира, но имеет дополнительные поля — доступ в лаунж, приоритетную посадку. Реализуйте одну и ту же задачу двумя способами: через наследование (BusinessPassenger(Passenger) — переопределяет конструктор и добавляет свои поля) и через композицию (BusinessPassenger содержит объект Passenger внутри себя как поле self.passenger). Сравните плюсы и минусы каждого подхода: наследование даёт переиспользование кода, но создаёт жёсткую связь; композиция гибче, но требует больше кода. Объясните, когда что выбирать, и приведите примеры из реальных проектов и стандартной библиотеки Python.

In [ ]:
class Passenger:
    def __init__(self, name):
        self.name = name

    def show_info(self):
        print("Пассажир:", self.name)


class BusinessPassenger(Passenger):
    def __init__(self, name, lounge_access, priority_boarding):
        super().__init__(name)
        self.lounge_access = lounge_access
        self.priority_boarding = priority_boarding


business = BusinessPassenger("Анна", True, True)

business.show_info()
print("Доступ в лаунж:", business.lounge_access)
print("Приоритетная посадка:", business.priority_boarding)
print("Является Passenger:", isinstance(business, Passenger))

Пассажир: Анна
Доступ в лаунж: True
Приоритетная посадка: True
Является Passenger: True


In [ ]:
class Passenger:
    def __init__(self, name):
        self.name = name

    def show_info(self):
        print("Пассажир:", self.name)


class BusinessPassenger:
    def __init__(self, passenger, lounge_access, priority_boarding):
        self.passenger = passenger
        self.lounge_access = lounge_access
        self.priority_boarding = priority_boarding

    def show_info(self):
        self.passenger.show_info()


passenger = Passenger("Анна")
business = BusinessPassenger(passenger, True, True)

business.show_info()
print("Доступ в лаунж:", business.lounge_access)
print("Приоритетная посадка:", business.priority_boarding)
print("Является Passenger:", isinstance(business, Passenger))

Пассажир: Анна
Доступ в лаунж: True
Приоритетная посадка: True
Является Passenger: False


Задача 29 (копирование объектов). В аэропорту нужно создавать копии посадочных талонов — например, для резервного архива. При этом у пассажира есть вложенный объект Luggage, и важно понимать, копируется ли он вместе с пассажиром или остаётся общим. Реализуйте в классе Passenger методы __copy__ и __deepcopy__. Создайте пассажира с вложенным объектом Luggage и продемонстрируйте разницу между поверхностным (copy.copy) и глубоким (copy.deepcopy) копированием: при поверхностном — багаж общий (shallow.luggage is passenger.luggage → True), при глубоком — свой (deep.luggage is passenger.luggage → False). Объясните, когда какой тип копирования использовать и что делать с циклическими ссылками.

In [ ]:
import copy


class Luggage:
    def __init__(self, weight):
        self.weight = weight
        self.owner = None


class Passenger:
    def __init__(self, name, luggage):
        self.name = name
        self.luggage = luggage

    def __copy__(self):
        return Passenger(self.name, self.luggage)

    def __deepcopy__(self, memo):
        new_passenger = Passenger(self.name, None)
        memo[id(self)] = new_passenger
        new_passenger.luggage = copy.deepcopy(self.luggage, memo)
        return new_passenger


passenger = Passenger("Анна", Luggage(20))

shallow = copy.copy(passenger)
deep = copy.deepcopy(passenger)

print("shallow is passenger:", shallow is passenger)
print("deep is passenger:", deep is passenger)
print("shallow.luggage is passenger.luggage:", shallow.luggage is passenger.luggage)
print("deep.luggage is passenger.luggage:", deep.luggage is passenger.luggage)

shallow.luggage.weight = 25

print("\nВес багажа после изменения через shallow:")
print("Оригинал:", passenger.luggage.weight)
print("Поверхностная копия:", shallow.luggage.weight)
print("Глубокая копия:", deep.luggage.weight)

cyclic = Passenger("Иван", Luggage(10))
cyclic.luggage.owner = cyclic

cyclic_copy = copy.deepcopy(cyclic)
print("\nЦикл ведёт к копии:", cyclic_copy.luggage.owner is cyclic_copy)

shallow is passenger: False
deep is passenger: False
shallow.luggage is passenger.luggage: True
deep.luggage is passenger.luggage: False

Вес багажа после изменения через shallow:
Оригинал: 25
Поверхностная копия: 25
Глубокая копия: 20

Цикл ведёт к копии: True


Задача 30 (утиная типизация). В аэропорту есть разные объекты, у которых есть метод describe() — пассажиры, пилоты, рейсы, а может быть, и что-то ещё, что появится в будущем. Напишите функцию describe_member(member), которая вызывает метод describe() у любого объекта. Продемонстрируйте, что функция работает не только с Passenger, но и с любым другим классом, у которого есть describe() (например, классом Flight), — без наследования от общего родителя. Объясните принцип утиной типизации: «если это выглядит как утка, плавает как утка и крякает как утка — это утка». Расскажите, как этот принцип отличает Python от языков со строгой типизацией (Java, C++) и какие у него минусы (ошибка видна только во время выполнения).

In [2]:
class Passenger:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"Пассажир: {self.name}"


class Flight:
    def __init__(self, number):
        self.number = number

    def describe(self):
        return f"Рейс: {self.number}"


def describe_member(member):
    return member.describe()


passenger = Passenger("Анна")
flight = Flight("SU123")

print(describe_member(passenger))
print(describe_member(flight))

Пассажир: Анна
Рейс: SU123


Задача 31 (итерация по объекту: __iter__ и __next__). В аэропорту есть рейс Flight, и хочется перебирать пассажиров на нём в цикле for — так же удобно, как список. Сейчас for passenger in flight: не работает. Реализуйте в классе Flight магические методы __iter__ и __next__ так, чтобы по рейсу можно было итерироваться несколько раз подряд (это важно: __iter__ должен возвращать отдельный объект-итератор или генератор, а не self, иначе повторная итерация сломается). Создайте рейс с несколькими пассажирами и переберите их в цикле for дважды подряд. Объясните, чем отличается итератор от итерируемого объекта, зачем нужны оба метода и почему list(flight) должен возвращать список всех пассажиров.

In [ ]:
class FlightIterator:
    def __init__(self, passengers):
        self.passengers = passengers
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.passengers):
            raise StopIteration

        passenger = self.passengers[self.index]
        self.index += 1
        return passenger


class Flight:
    def __init__(self, number, passengers):
        self.number = number
        self.passengers = passengers

    def __iter__(self):
        return FlightIterator(self.passengers)


flight = Flight("SU123", ["Анна", "Иван", "Мария"])

print("Первый обход:")
for passenger in flight:
    print(passenger)

print("Второй обход:")
for passenger in flight:
    print(passenger)

print("Список пассажиров:")
print(list(flight))

Задача 32 (индексация объекта: __getitem__ и __setitem__). На рейсе пассажиры должны быть доступны по индексу — как элементы списка: flight[0] возвращает первого пассажира, flight[1] = passenger заменяет второго. Реализуйте в классе Flight магические методы __getitem__ и __setitem__. Продемонстрируйте, что теперь работают: flight[0] — получение пассажира по индексу, flight[1] = new_passenger — замена пассажира, срезы flight[0:2] — получение подсписка, for passenger in flight — итерация (за счёт __getitem__, даже без __iter__). Дополнительно реализуйте __delitem__ (удаление по индексу) и __contains__ (проверка passenger in flight). Объясните, как __getitem__ делает объект «похожим» на список или словарь и почему важно, чтобы он выбрасывал именно IndexError, а не какое-то своё исключение.

In [3]:

class Flight:
    def __init__(self, passengers):
        self.passengers = passengers

    def __getitem__(self, index):
        return self.passengers[index]

    def __setitem__(self, index, passenger):
        self.passengers[index] = passenger

    def __delitem__(self, index):
        del self.passengers[index]

    def __contains__(self, passenger):
        return passenger in self.passengers


flight = Flight(["Иван", "Анна", "Пётр"])

print("Первый:", flight[0])

flight[1] = "Мария"
print("После замены:", flight.passengers)

print("Срез:", flight[0:2])

print("Все пассажиры:")
for passenger in flight:
    print(passenger)

print("Иван на рейсе?", "Иван" in flight)

del flight[0]
print("После удаления:", flight.passengers)


Первый: Иван
После замены: ['Иван', 'Мария', 'Пётр']
Срез: ['Иван', 'Мария']
Все пассажиры:
Иван
Мария
Пётр
Иван на рейсе? True
После удаления: ['Мария', 'Пётр']


Задача 33 (контекстный менеджер: __enter__ и __exit__). В аэропорту есть операции, которые нужно выполнять «аккуратно»: например, при работе с базой данных рейсов её нужно открыть в начале и обязательно закрыть в конце — даже если произошла ошибка. Реализуйте класс FlightDatabase как контекстный менеджер: методы __enter__ (открывает базу и возвращает её) и __exit__ (закрывает базу, даже если было исключение). Продемонстрируйте использование через with FlightDatabase() as db:. Дополнительно покажите, что происходит, если внутри with возникнет исключение, — база всё равно закроется. Объясните, зачем нужны контекстные менеджеры, как они связаны с конструкцией try/finally, что делают аргументы exc_type, exc_value, traceback в __exit__ и что нужно вернуть, чтобы «проглотить» исключение.

In [4]:

class FlightDatabase:
    def __enter__(self):
        print("База открыта")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("База закрыта")
        return False


with FlightDatabase() as db:
    print("Работаем с рейсами")

print("Проверка с ошибкой:")

try:
    with FlightDatabase() as db:
        print("Работаем с базой")
        raise ValueError("Ошибка в базе")
except ValueError as error:
    print("Поймали ошибку:", error)


База открыта
Работаем с рейсами
База закрыта
Проверка с ошибкой:
База открыта
Работаем с базой
База закрыта
Поймали ошибку: Ошибка в базе


Задача 34 (паттерн Strategy + super()). В аэропорту есть разные способы расчёта стоимости багажа: фиксированная (1000 ₽ за место), по весу (500 ₽ за каждый килограмм свыше 20 кг) и «эконом» (для эконом-класса — фиксированная, но с доплатой за перевес). Реализуйте паттерн «Стратегия»: базовый класс BaggageCostStrategy с методом calculate(weight), и три наследника — FixedBaggageCost, WeightBaggageCost, EconomyBaggageCost. Каждый наследник переопределяет calculate. Дополнительно используйте super() в классе EconomyBaggageCost, чтобы вызвать реализацию FixedBaggageCost и добавить надбавку за перевес. Продемонстрируйте работу всех трёх стратегий через единый интерфейс. Объясните, что такое паттерн «Стратегия», зачем он нужен, как он убирает длинные цепочки if/elif и как super() помогает переиспользовать код родителя.

In [ ]:

class BaggageCostStrategy:
    def calculate(self, weight):
        raise NotImplementedError


class FixedBaggageCost(BaggageCostStrategy):
    def calculate(self, weight):
        return 1000


class WeightBaggageCost(BaggageCostStrategy):
    def calculate(self, weight):
        return max(0, weight - 20) * 500


class EconomyBaggageCost(FixedBaggageCost):
    def calculate(self, weight):
        base = super().calculate(weight)
        extra = max(0, weight - 20) * 500
        return base + extra


strategies = [
    FixedBaggageCost(),
    WeightBaggageCost(),
    EconomyBaggageCost()
]

for strategy in strategies:
    print(type(strategy).__name__, ":", strategy.calculate(25), "руб.")


Задача 35 (принципы OCP и LSP из SOLID). В аэропорту появился новый тип участника — бортпроводник (FlightAttendant). Если добавлять его через изменение базового класса AirportMember, это нарушит принцип OCP (Open/Closed Principle) — классы должны быть открыты для расширения, но закрыты для изменения. Дополнительно нужно проверить принцип LSP (Liskov Substitution Principle) — объект наследника должен без проблем заменять объект родителя. Реализуйте: базовый класс AirportMember с методом get_status(); наследников Pilot, Passenger; функцию process_member(member: AirportMember), которая работает с любым участником через родительский интерфейс. Добавьте FlightAttendant без изменения существующего кода — это и есть OCP. Затем проверьте, что process_member одинаково хорошо работает со всеми тремя типами, — это LSP. Объясните оба принципа и приведите примеры их нарушения (классический пример нарушения LSP: «квадрат — не прямоугольник», если сеттеры ширины и высоты меняют оба измерения).

In [ ]:

class AirportMember:
    def get_status(self):
        return "Участник аэропорта"


class Pilot(AirportMember):
    def get_status(self):
        return "Пилот"


class Passenger(AirportMember):
    def get_status(self):
        return "Пассажир"


def process_member(member: AirportMember):
    print("Статус:", member.get_status())


# Новый класс без изменения старых классов
class FlightAttendant(AirportMember):
    def get_status(self):
        return "Бортпроводник"


members = [Pilot(), Passenger(), FlightAttendant()]

for member in members:
    process_member(member)


Задача 36 (объект как функция: __call__). В аэропорту есть служебный объект TicketCalculator, который должен уметь рассчитывать итоговую стоимость билета с учётом сборов. Удобно, чтобы его можно было вызывать как функцию: calculator(base_price), а не calculator.calculate(base_price). Реализуйте в классе TicketCalculator магический метод __call__(self, base_price), который возвращает итоговую стоимость (например, base_price + 2000 — топливный сбор). Продемонстрируйте, что объект можно вызывать как функцию: calc = TicketCalculator(); calc(10000). Дополнительно покажите, что такой объект можно передать туда, где ожидается функция-колбэк. Объясните, зачем нужен __call__, чем он полезен (функторы, декораторы-классы, колбэки с состоянием) и как он связан с понятием «функтор» в других языках.

In [5]:

class TicketCalculator:
    def __init__(self):
        self.fee = 2000

    def __call__(self, base_price):
        return base_price + self.fee


def show_price(callback, price):
    print("Итоговая цена:", callback(price))


calc = TicketCalculator()

print("Цена:", calc(10000))

show_price(calc, 15000)


Цена: 12000
Итоговая цена: 17000


Задача 37 (__new__ vs __init__). В аэропорту есть класс Passenger, и нужно понять, в каком порядке Python создаёт объект. Реализуйте в классе Passenger оба метода — __new__ и __init__ — и добавьте в каждый отладочный вывод (print), чтобы увидеть порядок вызова. Продемонстрируйте, что __new__ вызывается до __init__ и отвечает за создание объекта, а __init__ — за его инициализацию. Дополнительно реализуйте __new__, который возвращает уже существующий объект (например, для кэширования пассажиров по имени и номеру рейса): два вызова Passenger("Иван Петров", "SU1234", "12A") должны вернуть один и тот же объект. Объясните разницу между __new__ и __init__, зачем вообще нужен __new__ и почему «__new__ + кэш» легко приводит к странному поведению.

In [ ]:

class Passenger:
    cache = {}

    def __new__(cls, name, flight_number, seat):
        print("Вызван __new__")
        key = (name, flight_number)

        if key not in cls.cache:
            cls.cache[key] = super().__new__(cls)

        return cls.cache[key]

    def __init__(self, name, flight_number, seat):
        print("Вызван __init__")

        if not hasattr(self, "name"):
            self.name = name
            self.flight_number = flight_number
            self.seat = seat


p1 = Passenger("Иван Петров", "SU1234", "12A")
p2 = Passenger("Иван Петров", "SU1234", "12A")

print("Один объект?", p1 is p2)
print("Место:", p2.seat)


Задача 38 (паттерн Decorator). В аэропорту есть базовый класс AirportMember с методом get_info(). Нужно добавлять к участнику дополнительную информацию — например, «VIP», «с животным», «нужна помощь при посадке». При этом нельзя менять сам класс AirportMember. Реализуйте паттерн «Декоратор»: класс MemberDecorator, который содержит объект AirportMember внутри себя и имеет тот же метод get_info(), но добавляет к результату дополнительный текст. Создайте декораторы VipDecorator и AssistanceDecorator, которые можно комбинировать: VipDecorator(AssistanceDecorator(passenger)). Продемонстрируйте работу, а также покажите, что смена порядка декораторов меняет порядок меток в выводе. Объясните, чем паттерн «Декоратор» отличается от наследования и почему он удобнее, когда нужно много комбинаций украшений (иначе пришлось бы создавать 2^n подклассов).

In [ ]:

class AirportMember:
    def __init__(self, name):
        self.name = name

    def get_info(self):
        return self.name


class MemberDecorator:
    def __init__(self, member):
        self.member = member

    def get_info(self):
        return self.member.get_info()


class VipDecorator(MemberDecorator):
    def get_info(self):
        return super().get_info() + " | VIP"


class AssistanceDecorator(MemberDecorator):
    def get_info(self):
        return super().get_info() + " | Нужна помощь"


passenger = AirportMember("Иван")

first = VipDecorator(AssistanceDecorator(passenger))
second = AssistanceDecorator(VipDecorator(passenger))

print(first.get_info())
print(second.get_info())


Задача 39 (паттерн Adapter). В аэропорту появилась внешняя система бронирования, которая возвращает данные о пассажирах в другом формате — например, в виде словаря {"full_name": "Иван Петров", "flight": "SU1234", "seat_number": "12A"}. А весь остальной код аэропорта работает с классом Passenger, у которого поля name, flight_number, seat. Реализуйте паттерн «Адаптер»: класс PassengerAdapter, который принимает словарь и предоставляет интерфейс Passenger — свойства name, flight_number, seat и метод info(). Продемонстрируйте, что адаптер можно использовать там, где ожидается Passenger — например, передать его в функцию, которая работает с пассажиром. Объясните, что такое паттерн «Адаптер», зачем он нужен, чем отличается от Facade и Decorator, и как он помогает интегрировать несовместимые интерфейсы без изменения существующего кода.

In [ ]:

class Passenger:
    def __init__(self, name, flight_number, seat):
        self.name = name
        self.flight_number = flight_number
        self.seat = seat

    def info(self):
        return f"{self.name}, рейс {self.flight_number}, место {self.seat}"


class PassengerAdapter:
    def __init__(self, data):
        self.data = data

    @property
    def name(self):
        return self.data["full_name"]

    @property
    def flight_number(self):
        return self.data["flight"]

    @property
    def seat(self):
        return self.data["seat_number"]

    def info(self):
        return f"{self.name}, рейс {self.flight_number}, место {self.seat}"


def show_passenger(passenger):
    print(passenger.info())


data = {
    "full_name": "Иван Петров",
    "flight": "SU1234",
    "seat_number": "12A"
}

passenger = PassengerAdapter(data)
show_passenger(passenger)


Задача 40 (принципы ISP и DIP из SOLID). В аэропорту есть базовый класс AirportMember, и постепенно в нём накопились методы, которые нужны не всем участникам: fly() (только для пилотов), serve() (только для бортпроводников), board() (только для пассажиров). Это нарушает принцип ISP (Interface Segregation Principle) — клиент не должен зависеть от методов, которые он не использует. Разделите интерфейс на несколько маленьких: Flyable, Servable, Boardable, и пусть каждый класс реализует только нужные. Дополнительно проверьте принцип DIP (Dependency Inversion Principle) — модули высокого уровня не должны зависеть от модулей низкого уровня; оба должны зависеть от абстракций. Реализуйте класс BoardingService, который зависит не от конкретных классов (Pilot, Passenger), а от абстракции Boardable (или от Protocol). Продемонстрируйте, что можно легко добавить новый тип участника (например, FlightAttendant, реализующий только Servable) без изменения BoardingService. Объясните оба принципа, приведите примеры их нарушения и расскажите, почему DIP помогает тестировать код (легко подменять зависимости на mock).

In [6]:

from abc import ABC, abstractmethod


class Flyable(ABC):
    @abstractmethod
    def fly(self):
        pass


class Servable(ABC):
    @abstractmethod
    def serve(self):
        pass


class Boardable(ABC):
    @abstractmethod
    def board(self):
        pass


class Pilot(Flyable):
    def fly(self):
        print("Пилот управляет самолётом")


class Passenger(Boardable):
    def board(self):
        print("Пассажир садится в самолёт")


class FlightAttendant(Servable):
    def serve(self):
        print("Бортпроводник обслуживает пассажиров")


class BoardingService:
    def process(self, member: Boardable):
        member.board()


service = BoardingService()

pilot = Pilot()
passenger = Passenger()
attendant = FlightAttendant()

pilot.fly()
service.process(passenger)
attendant.serve()


Пилот управляет самолётом
Пассажир садится в самолёт
Бортпроводник обслуживает пассажиров
